Install Libraries

In [ ]:
!pip install pandas numpy scikit-learn xgboost seaborn matplotlib

Import Libraries

In [3]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report

from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from xgboost import XGBClassifier

import joblib

Upload Dataset

In [4]:
from google.colab import files
uploaded = files.upload()


Saving panic_dataset_cleaned.csv to panic_dataset_cleaned.csv


,Participant ID,Age,Gender,Family History,Personal History,Current Stressors,Symptoms,Severity,Impact on Life,Demographics,Medical History,Psychiatric History,Substance Use,Coping Mechanisms,Social Support,Lifestyle Factors,Panic Disorder Diagnosis,Panic attack
0,1,38,Male,No,Yes,Moderate,Shortness of breath,Mild,Mild,Rural,Diabetes,Bipolar disorder,Unknown,Socializing,High,Sleep quality,0,0
1,2,51,Male,No,No,High,Panic attacks,Mild,Mild,Urban,Asthma,Anxiety disorder,Drugs,Exercise,High,Sleep quality,0,1
2,3,32,Female,Yes,No,High,Panic attacks,Mild,Significant,Urban,Diabetes,Depressive disorder,Unknown,Seeking therapy,Moderate,Exercise,0,2
3,4,64,Female,No,No,Moderate,Chest pain,Moderate,Moderate,Rural,Diabetes,Unknown,Unknown,Meditation,High,Exercise,0,0
4,5,31,Male,Yes,No,Moderate,Panic attacks,Mild,Moderate,Rural,Asthma,Unknown,Drugs,Seeking therapy,Low,Sleep quality,0,2


In [ ]:

df = pd.read_csv("panic_dataset_cleaned.csv")
df.head()

Data Cleaning

In [5]:
# Drop ID column if exists
df = df.drop(columns=["Participant ID"], errors="ignore")

# Drop missing values
df = df.dropna()

In [ ]:
y = df["Panic attack"]

In [ ]:
tfidf = TfidfVectorizer(max_features=50)

symptoms_tfidf = tfidf.fit_transform(df["Symptoms"].astype(str)).toarray()

Encode Categorical Columns

In [6]:
le = LabelEncoder()

for col in df.columns:
    if df[col].dtype == "object" and col != "Symptoms":
        df[col] = le.fit_transform(df[col].astype(str))

In [ ]:
X_numeric = df.drop(columns=["Panic attack", "Symptoms"]).values

X = np.hstack([X_numeric, symptoms_tfidf])

Feature Engineering

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

Model 1 — Logistic Regression

In [ ]:
lr = LogisticRegression(max_iter=2000)

lr.fit(X_train, y_train)
pred_lr = lr.predict(X_test)

print("LR Accuracy:", accuracy_score(y_test, pred_lr))

Model 2 — Random Forest

In [ ]:
rf = RandomForestClassifier(n_estimators=300)

rf.fit(X_train, y_train)
pred_rf = rf.predict(X_test)

print("RF Accuracy:", accuracy_score(y_test, pred_rf))

Model 3 — SVM

In [ ]:
svm = SVC(probability=True)

svm.fit(X_train, y_train)
pred_svm = svm.predict(X_test)

print("SVM Accuracy:", accuracy_score(y_test, pred_svm))

Model 4 — XGBoost

In [ ]:
xgb = XGBClassifier(
    n_estimators=400,
    learning_rate=0.05,
    max_depth=6
)

xgb.fit(X_train, y_train)
pred_xgb = xgb.predict(X_test)

print("XGB Accuracy:", accuracy_score(y_test, pred_xgb))

Model Comparison

In [ ]:
results = pd.DataFrame({
    "Model": ["LR", "RF", "SVM", "XGB"],
    "Accuracy": [
        accuracy_score(y_test, pred_lr),
        accuracy_score(y_test, pred_rf),
        accuracy_score(y_test, pred_svm),
        accuracy_score(y_test, pred_xgb)
    ]
})

results

In [ ]:
best_model = xgb

In [ ]:
joblib.dump(best_model, "model.pkl")
joblib.dump(list(range(X.shape[1])), "features.pkl")  # feature indices
joblib.dump(tfidf, "tfidf.pkl")

In [ ]:
joblib.dump(best_model, "model.pkl")
joblib.dump(list(range(X.shape[1])), "features.pkl")  # feature indices
joblib.dump(tfidf, "tfidf.pkl")